要运行此程序，请在**免费** Tesla T4 Google Colab 实例上按“*运行时*”并按“*全部运行*”！
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> 如果您需要帮助，请加入 Discord + ⭐ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</a> </i> ⭐
</div>

要在本地设备上安装 Unsloth，请按照 [our guide](https://unsloth.ai/docs/get-started/install) 操作。此笔记本已获得许可 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)。

您将学习如何执行 [data prep](#Data)、如何执行 [train](#Train)、如何执行 [run the model](#Inference) 以及如何保存它

### 消息

隆重推出 **Unsloth Studio** - 一个新的开源、无代码 Web UI，用于训练和运行法学硕士。 [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<表><tr>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~% 2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fupload s%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia% 26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&宽度=376&dpr=3&质量=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>训练模型</b> - 无需代码</sub></td>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Z q%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26toke n%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&宽度=376&dpr=3&质量=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio 聊天 UI"></a><br><sub><b>在 Mac、Windows 和 Linux 上运行 GGUF 模型</b></sub></td>
</tr></表>

训练 MoE - DeepSeek、GLM、Qwen 和 gpt-oss 速度提高 12 倍，VRAM 减少 35%。 [Blog](https://unsloth.ai/docs/new/faster-moe)

超长上下文强化学习现已推出，上下文窗口增加了 7 倍！ [Blog](https://unsloth.ai/docs/new/grpo-long-context)

强化学习新功能：[FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

请访问我们的文档以获取所有 [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) 和 [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks)。

### 安装

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  #  Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### 不懒惰

`FastModel` 现在支持加载几乎任何模型！这包括视觉和文本模型！

In [1]:
from unsloth import FastModel
from transformers import AutoModelForSequenceClassification
import torch

# 禁用 bert 的快速生成！
%env UNSLOTH_DISABLE_FAST_GENERATION = 1

max_seq_length = 2048 # 选择任何一个！我们内部自动支持 RoPE Scaling！
dtype = None # 没有用于自动检测。 Float16 适用于 Tesla T4、V100，Bfloat16 适用于 Ampere+
load_in_4bit = False # 使用 4 位量化来减少内存使用。可能是假的。

# 我们支持 4 位预量化模型，下载速度提高 4 倍 + 无 OOM。
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 万亿代币模型速度提高 2 倍！
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",    # 我们还上传了 4bit 的 405b！
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # 新 Mistral 12b 速度提高 2 倍！
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 速度提高 2 倍！
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 速度提高 2 倍！
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # 杰玛速度快了 2 倍！
] # 更多模型请访问 https://huggingface.co/unsloth

id2label = {0: "sadness", 1: "joy", 2: "love", 3: "anger",4: "fear",5: "surprise"}

label2id = {"sadness": 0, "joy": 1, "love": 2, "anger": 3, "fear": 4, "surprise": 5}

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/ModernBERT-large",
    auto_model = AutoModelForSequenceClassification,
    max_seq_length = max_seq_length,
    dtype = dtype,
    num_labels  = 6,
    full_finetuning = True,
    id2label = id2label,
    label2id = label2id,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # 门控模型的 HF 令牌
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
env: UNSLOTH_DISABLE_FAST_GENERATION=1
==((====))==  Unsloth 2025.8.7: Fast Modernbert patching. Transformers: 4.55.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Float16 full finetuning uses more memory since we upcast weights to float32.

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

我们现在添加了 LoRA 适配器，因此我们只需要更新少量参数！

In [2]:
model = FastModel.get_peft_model(
    model,
    r = 16, # 选择任意大于 0 的数字！建议 8、16、32、64、128
      target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # 支持任意，但 = 0 已优化
    bias = "none",    # 支持任何，但=“无”已优化
    # [新]“unsloth”使用的 VRAM 减少了 30%，适合 2 倍大的批量大小！
    use_gradient_checkpointing = "unsloth", # 对于很长的上下文来说是真实的或“不懒惰的”
    random_state = 3407,
    use_rslora = False,  # 我们支持排名稳定的 LoRA
    loftq_config = None, # 和阁楼Q
    task_type = "SEQ_CLS",
)

Unsloth: Full finetuning is enabled, so .get_peft_model has no effect

<a名称=“数据”></a>
### 数据准备  
我们现在使用“dair-ai”中的 [Emotion dataset](https://huggingface.co/datasets/dair-ai/emotion)，其中包含由情感标记的文本。在此示例中，我们加载 **unsplit** 版本并仅使用前 30,000 个样本。  

然后，我们将数据集分为训练 (80%) 和验证 (20%)，并应用标记化来准备训练文本。

In [3]:
from datasets import load_dataset

# 加载 IMDB 数据集
dataset = load_dataset("dair-ai/emotion","unsplit",split = 'train[:30000]')

# 分为训练集和验证集
dataset = dataset.train_test_split(test_size = 0.2)

def tokenize_function(examples):
    return tokenizer(examples["text"])

# 将分词器应用到数据集
train_dataset = dataset['train'].map(tokenize_function, batched = True)
val_dataset = dataset["test"].map(tokenize_function, batched = True)

Map:   0%|          | 0/24000 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

我们使用 scikit-learn 的“compute_class_weight”来计算**类权重**。  
当对某些类别代表性不足的数据集进行训练时，这非常有用，可确保模型不会偏向大多数标签。

In [4]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = train_dataset["label"]
class_weights = compute_class_weight("balanced", classes = np.unique(labels), y = labels)

In [5]:
# 我们将数据集列从 **`label`** 重命名为 **`labels`**，因为这是 Hugging Face `Trainer` 的预期字段名称。
train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")

In [6]:
class_weights

array([0.56116723, 0.49943813, 1.998002  , 1.23839009, 1.45932142,
       4.49438202])

我们定义一个“compute_metrics”函数来在训练期间评估模型。  
在这里，我们使用 scikit-learn 中的**准确性**，它将预测标签与真实情况进行比较。  

**[注意]** 准确性是一个很好的基准，但对于不平衡的数据集，您可能还需要跟踪 **F1-score**、** precision** 或 **recall** 等指标。

In [7]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis = -1)
    return {"accuracy": accuracy_score(labels, preds)}

<a name="火车"></a>
### 训练模型
现在让我们使用拥抱脸部“训练器”！更多文档在这里：[Transformers docs](https://huggingface.co/docs/transformers/main_classes/trainer)。我们训练一个完整的 epoch (num_train_epochs=1) 以获得有意义的结果。

In [8]:
from transformers import TrainingArguments,Trainer
from unsloth import is_bfloat16_supported

trainer = Trainer(
    model = model,
    processing_class = tokenizer,
    eval_dataset = val_dataset,
    train_dataset = train_dataset,
    args = TrainingArguments(
        per_device_train_batch_size = 32,
        gradient_accumulation_steps = 1,
        warmup_steps = 5,
        num_train_epochs = 1, # bert 风格的模型通常需要 1 个以上的 epoch
        # 最大步数 = 60,
        learning_rate = 5e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        eval_strategy = "steps",
        eval_steps = 0.10,  # 评估总训练步骤的每 10%
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # 使用 TrackIO/WandB 等
    ),
    compute_metrics = compute_metrics,
)

让我们训练模型吧！要恢复训练运行，请设置“trainer.train(resume_from_checkpoint = True)”

In [9]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 24,000 | Num Epochs = 1 | Total steps = 750
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 395,837,446 of 395,837,446 (100.00% trained)


Step,Training Loss,Validation Loss,Accuracy
75,1.559700,1.585444,0.347000
150,1.435100,1.583412,0.348833
225,1.741100,1.569451,0.344167
300,1.167800,1.416889,0.494333
375,1.342500,1.258238,0.541167
450,1.120800,1.076675,0.593167
525,0.860900,0.940197,0.651667
600,0.712700,0.820846,0.715500
675,0.757300,0.729059,0.748833
750,0.915100,0.688935,0.758167


Unsloth: Will smartly offload gradients to save VRAM!


<a name="推理"></a>
### 推论
让我们运行模型吧！

In [45]:
from transformers import pipeline

sentence1 = "We just finished training ModernBERT with Unsloth and it's amazing!"

classifier = pipeline("sentiment-analysis", model = model,tokenizer = tokenizer)

classifier(sentence1)

Device set to use cuda:0

[{'label': 'joy', 'score': 0.7757943272590637}]

<a name="保存"></a>
### 保存微调模型
要保存最终模型，请使用 Hugging Face 的 `push_to_hub` 进行在线保存，或使用 `save_pretrained` 进行本地保存。

In [15]:
model.save_pretrained("bert_classification_lora")  # 本地储蓄
tokenizer.save_pretrained("bert_classification_lora")
# model.push_to_hub("your_name/bert_classification_lora", token = "YOUR_HF_TOKEN") # 在线保存
# tokenizer.push_to_hub("your_name/bert_classification_lora", token = "YOUR_HF_TOKEN") # 在线保存

('model/tokenizer_config.json',
 'model/special_tokens_map.json',
 'model/tokenizer.json')

我们就完成了！如果您对 Unsloth 有任何疑问，我们有 [Discord](https://discord.gg/unsloth) 频道！如果您发现任何错误或想要随时更新最新的 LLM 内容，或需要帮助、加入项目等，请随时加入我们的 Discord！

其他一些资源：
1.训练自己的推理模型——Llama GRPO笔记本[Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. 将微调保存到Ollama。 [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 视觉微调 - 射线照相用例。 [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. 请参阅我们的 [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks) 上的 DPO、ORPO、持续预训练、对话微调等笔记本！

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  如果您需要帮助，请加入 Discord + ⭐️ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</i> ⭐️
</div>

  此笔记本和所有 Unsloth 笔记本均已获得 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme) 许可。